In [34]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline

In [7]:
# data
iris = load_iris()
X, y = pd.DataFrame(iris.data, columns=iris.feature_names), pd.Series(iris.target, name='species')

# split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y) 

# pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(solver='lbfgs', max_iter=1000, random_state=42))
])

# for name, step in pipeline.named_steps.items():
#     print(f"  {name}: {step.__class__.__name__}")

# model
model = pipeline.fit(X_train, y_train)

# predict
y_pred = model.predict(X_test)
y_pred_train = model.predict(X_train)
y_pred_proba = model.predict_proba(X_test)

In [31]:
target_names = iris.target_names

print("\n" + "="*60)
print("MODEL EVALUATION")
print("="*60)

# accuracy
test_accuracy = accuracy_score(y_test, y_pred)
train_accuracy = accuracy_score(y_train, y_pred_train)
gap = train_accuracy - test_accuracy
print(f"Training Accuracy: {train_accuracy:.2%}")
print(f"Test Accuracy: {test_accuracy:.2%}")
print(f"Overfitting Gap: {gap:.4f}")


if gap < 0.01:
    print("✅ NO OVERFITTING (gap < 1%)")
    status = "Balanced"
elif gap < 0.02:
    print("✅ MINIMAL OVERFITTING (1-2%)")
    status = "Slight Overfitting"
elif gap < 0.03:
    print("⚠️  MILD OVERFITTING (2-3%)")
    status = "Mild Overfitting"
elif gap < 0.05:
    print("⚠️  MODERATE OVERFITTING (3-5%)")
    status = "Moderate Overfitting"
elif gap < 0.10:
    print("🔴 SIGNIFICANT OVERFITTING (5-10%)")
    status = "Significant Overfitting"
else:
    print("🔴 SEVERE OVERFITTING (>10%)")
    status = "Severe Overfitting"

if train_accuracy < 0.70 and test_accuracy < 0.70:
    print("📉 UNDERFITTING DETECTED: Model is too simple")
elif train_accuracy < 0.75:
    print("⚠️  Possible underfitting (training accuracy < 75%)")
else:
    print("✅ No underfitting detected")

# detailed classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=target_names))

# confusion matrix
cm = confusion_matrix(y_test, y_pred)
correct_predictions = np.trace(cm)
total_predictions = np.sum(cm)
accuracy = correct_predictions / total_predictions

print("\nConfusion Matrix:")
print(cm)


print(f"Correct predictions: {correct_predictions}")
print(f"Total predictions: {total_predictions}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Accuracy: {accuracy*100:.2f}%")
print()

for i in range(cm.shape[0]):
    class_correct = cm[i, i]
    class_total = np.sum(cm[i, :])
    accuracy = class_correct / class_total
    print(f"{target_names[i]} Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")


cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5)

print("\n" + "="*60)
print("CROSS-VALIDATION")
print("="*60)
print(f"CV Scores: {cv_scores}")
print(f"CV Mean: {cv_scores.mean():.4f}")
print(f"CV Std: {cv_scores.std():.4f}")


MODEL EVALUATION
Training Accuracy: 95.83%
Test Accuracy: 93.33%
Overfitting Gap: 0.0250
⚠️  MILD OVERFITTING (2-3%)
✅ No underfitting detected

Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.90      0.90      0.90        10
   virginica       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30


Confusion Matrix:
[[10  0  0]
 [ 0  9  1]
 [ 0  1  9]]
Correct predictions: 28
Total predictions: 30
Accuracy: 0.9333
Accuracy: 93.33%

setosa Accuracy: 1.0000 (100.00%)
versicolor Accuracy: 0.9000 (90.00%)
virginica Accuracy: 0.9000 (90.00%)

CROSS-VALIDATION
CV Scores: [0.91666667 0.95833333 0.95833333 0.95833333 1.        ]
CV Mean: 0.9583
CV Std: 0.0264


In [27]:
scaler = pipeline.named_steps['scaler']
model = pipeline.named_steps['classifier']

print("\n" + "="*60)
print("COEFICIENTS")
print("="*60)
print(f"Scaler mean: {scaler.mean_.round(3)}")
print(f"Scaler std: {scaler.scale_.round(3)}")

# Model coefficients
coef_df = pd.DataFrame(
    model.coef_,
    columns=iris.feature_names,
    index=iris.target_names
)

columns = coef_df.columns
for col in columns:
    coef_df[f'odds_ratio_{col}'] = np.exp(coef_df[f'{col}'])

print("\nCoefficients:")
coef_df


ACCESSING PIPELINE COMPONENTS
Scaler mean: [5.842 3.048 3.77  1.205]
Scaler std: [0.837 0.447 1.761 0.759]

Coefficients:


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),odds_ratio_sepal length (cm),odds_ratio_sepal width (cm),odds_ratio_petal length (cm),odds_ratio_petal width (cm)
setosa,-1.088945,1.024208,-1.799056,-1.686228,0.336571,2.784888,0.165455,0.185217
versicolor,0.536337,-0.360487,-0.204074,-0.807957,1.709732,0.697337,0.815402,0.445768
virginica,0.552608,-0.663721,2.003130,2.494185,1.737780,0.514932,7.412222,12.111861


In [35]:
print("\n" + "="*60)
print("HYPERPARAMETER TUNING WITH PIPELINE")
print("="*60)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=14)

param_grid = {
    'classifier__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'classifier__solver': ['lbfgs', 'saga'],
    'classifier__max_iter': [500, 1000]
}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best CV Score: {grid_search.best_score_:.4f}")

# Get best pipeline
best_pipeline = grid_search.best_estimator_
y_pred_best = best_pipeline.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred_best)

print(f"Test Accuracy with best params: {test_accuracy:.4f}")


HYPERPARAMETER TUNING WITH PIPELINE
Fitting 5 folds for each of 24 candidates, totalling 120 fits
Best Parameters: {'classifier__C': 1, 'classifier__max_iter': 500, 'classifier__solver': 'lbfgs'}
Best CV Score: 0.9667
Test Accuracy with best params: 0.9333


In [37]:
# Regularization Strength = 1 / C
# Small C (e.g., 0.01) = Strong regularization (simpler model)
# Large C (e.g., 100) = Weak regularization (more complex model)
# C = 1 = Default, balanced regularization